In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def calc_out(w, f, s, p):
    print(f'Output Shape: {(w - f + 2 * p) // s + 1}')

# Conv1 (kernel3, stride1, pad1) — preserves size
calc_out(28, 3, 1, 1)   # → 28

# MaxPool1 (kernel2, stride2, padding=1) — given in the diagram
calc_out(28, 2, 2, 1)   # → 15

# Conv2, Conv3, Conv4 (kernel3, stride1, pad1) — each preserves size
calc_out(15, 3, 1, 1)   # → 15

# MaxPool2 (kernel2, stride2, padding=0)
calc_out(15, 2, 2, 0)   # → 7

Output Shape: 28
Output Shape: 15
Output Shape: 15
Output Shape: 7


In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3,3), stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3,3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3,3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3,3), stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=0)

        self.dropout = nn.Dropout(p=0.2)

        self.fcn1 = nn.Linear(256*7*7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = x.view(-1, 256*7*7)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x

In [4]:
model = CNN(num_classes=10)
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

In [5]:
dum = torch.randn(4, 1, 28, 28)
out = model(dum)
print(out.shape)

torch.Size([4, 10])
